**News Sentiment as a Trading Signal: Measuring Predictive Decay Across Holding Horizons**

Syed Sirajuddin · Master of Science in Applied Artificial Intelligence · Shiley Marcos School of Engineering, University of San Diego · AAI-590 Capstone

# Notebook 5 of 5 — Optimization, Analysis, and Discussion

This notebook contains the Model Optimization and the Analysis/Discussion elements of the code base, feeding the Methodology (optimization procedure), Results, and Conclusion sections of the report. The optimization *procedure* is fully specified and implemented here; the resulting numbers, figures, and their discussion are placeholders until model training and testing complete, per the project schedule.


In [ ]:
# Environment setup: resolve the repository root so `src` imports work
# whether this notebook is run from notebooks/ or the project root.
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
RANDOM_SEED = 42

## 1. Optimization Procedure

Three signal hyperparameters are tuned: the sentiment entry threshold, the rolling aggregation window, and the minimum article support. The grid is deliberately coarse (three to four values per parameter) and the protocol strict, because hyperparameter search over backtests is the canonical route to overfitting — with enough trials, a "profitable" configuration emerges from noise alone (Bailey et al., 2014). Three rules keep the search honest: (1) the grid is evaluated **only on the in-sample period** (before January 2024); (2) the single configuration selected in-sample is then evaluated **once** on the out-of-sample window, and that number is final; and (3) the selection criterion is in-sample Sharpe at the *median* horizon rather than the best horizon, so the horizon-decay result is not contaminated by having been selected on.


In [ ]:
from itertools import product
from src.config import PROCESSED_DIR, UNIVERSE, BACKTEST, SIGNALS
from src.features.aggregate import daily_sentiment_panel
from src.features.technicals import add_technicals
from src.signals.generate import generate_signals
from src.backtest.engine import run_horizon
from src.backtest.metrics import sharpe_ratio

prices = pd.read_parquet(PROCESSED_DIR / "prices_clean.parquet")
scored = pd.read_parquet(PROCESSED_DIR / "news_scored.parquet")
trading_days = pd.DatetimeIndex(prices["date"].drop_duplicates().sort_values())
prices_feat = add_technicals(prices)
oos = pd.Timestamp(BACKTEST.oos_start)
prices_is = prices[prices["date"] < oos]

GRID = {"long_threshold": [0.25, 0.35, 0.45],
        "sentiment_window": [1, 3, 5],
        "min_articles": [1, 2, 3]}
SELECTION_HORIZON = 10  # median of the horizon grid

rows = []
for thr, win, min_a in product(*GRID.values()):
    SIGNALS.long_threshold, SIGNALS.short_threshold = thr, -thr
    SIGNALS.sentiment_window, SIGNALS.min_articles = win, min_a
    panel = daily_sentiment_panel(scored, trading_days, UNIVERSE.tickers)
    sigs = generate_signals(panel, prices_feat)
    sigs_is = sigs[sigs["date"] < oos]
    res = run_horizon(sigs_is, prices_is, SELECTION_HORIZON)
    rows.append({"threshold": thr, "window": win, "min_articles": min_a,
                 "n_trades": len(res.trades),
                 "is_sharpe": sharpe_ratio(res.daily_returns)})

grid_results = pd.DataFrame(rows).sort_values("is_sharpe", ascending=False)
grid_results.head(10)

**Selected configuration — to be completed after the in-sample search.** The chosen values will be fixed in `src/config.py`, and every subsequent number in this notebook will use them.

## 2. Optional Learned Signal: LSTM Combiner

As a model-comparison extension (scheduled as the first item to be cut if time runs short), a recurrent sequence model combines rolling sentiment with price context to *learn* the entry signal rather than hard-coding thresholds. The architecture is a stacked LSTM (Hochreiter & Schmidhuber, 1997) — two recurrent layers of 64 and 32 units with dropout 0.2 between them, followed by a 16-unit ReLU layer and a sigmoid output — trained with binary cross-entropy and Adam (10⁻³) to predict the sign of the forward return from 20-day sequences of six features. Two leakage controls mirror the backtest discipline: a strictly chronological train/test split at the out-of-sample boundary, and feature normalization using training-period statistics only. Comparing the learned signal's decay curve to the rule-based one tests whether the decay is a property of the *information* or of our particular rule.


In [ ]:
from src.models.sequence import train, build_dataset

panel = daily_sentiment_panel(scored, trading_days, UNIVERSE.tickers)
model, history, oos_preds = train(panel, prices_feat, horizon=5)
model.summary()

**LSTM training curves and out-of-sample AUC — to be completed after training.**

## 3. Results (placeholder)

To be completed after live-data testing. This section will present, for the selected configuration and each holding horizon on the out-of-sample window: the horizon comparison table (trade counts, hit rate, average net trade return, Sharpe ratio, maximum drawdown) with the random-null 95th-percentile band; the decay-curve figure; the post-signal event study; and equity curves against buy-and-hold. The discussion will state plainly at which horizons, if any, the strategy's Sharpe exceeds the null band, and how transaction costs reshape the gross-signal decay.


In [ ]:
# Final out-of-sample evaluation — executed exactly once after the
# configuration is frozen. See Notebook 04 for the engine and baselines.
from src.backtest.engine import run_all_horizons
from src.backtest.baselines import buy_and_hold, random_signals_null
from src.analysis.horizon_decay import horizon_table, plot_decay_curve

sigs = generate_signals(panel, prices_feat)
sigs_oos = sigs[sigs["date"] >= oos]
results = run_all_horizons(sigs_oos, prices)
nulls = {h: random_signals_null(sigs_oos, prices, h, n_sims=100)
         for h in BACKTEST.horizons}
final_table = horizon_table(results, nulls)
final_table.round(3)

## 4. Conclusion (placeholder)

To be completed after Results. This section will summarize the hypothesis test — whether sentiment-driven signals beat the baselines at swing horizons and lose their edge as the holding period lengthens — highlight the most significant or unexpected finding, and outline future work: expanding the universe beyond large caps, richer text inputs (full articles, earnings-call transcripts), regime-conditional analysis, and the productionization path (streaming ingestion, scheduled scoring, and monitoring for sentiment-model drift).


---
### Acknowledgment of AI Tool Use

Portions of the code scaffolding and prose in this notebook were drafted with the assistance of Anthropic's Claude (Anthropic, 2026) and subsequently reviewed, tested, and revised by the author, who takes full responsibility for the final content, design decisions, and results. This acknowledgment is provided in accordance with University of San Diego academic integrity guidelines on the use of generative AI tools.

Anthropic. (2026). *Claude* [Large language model]. https://claude.ai

### References

Bailey, D. H., Borwein, J. M., López de Prado, M., & Zhu, Q. J. (2014). Pseudo-mathematics and financial charlatanism: The effects of backtest overfitting on out-of-sample performance. *Notices of the American Mathematical Society, 61*(5), 458–471.

Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation, 9*(8), 1735–1780.
